# Carnet Study Evaluation

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from datetime import datetime
from matplotlib import pyplot as plt
%matplotlib inline

In [ ]:
WITH_SERVICE_AUTHENTICATION = 'WITH_SERVICE_AUTHENTICATION'
WITH_CLIENT_AUTHENTICATION = 'WITH_CLIENT_AUTHENTICATION'
NO_SOMEIP_SD = 'NO_SOMEIP_SD'
WITH_DNSSEC = 'WITH_DNSSEC'
WITH_DANE = 'WITH_DANE'
WITH_ENCRYPTION = 'WITH_ENCRYPTION'

STATISTICS_PARENT_PATH='/home/vm-user/workspace/mininet-vsomeip-evaluation/carnet/statistic-results'

compile_definitions = { 'A':'Vanilla',
                        # 'B':f'{NO_SOMEIP_SD} {WITH_DNSSEC}',
                        # 'C':f'{WITH_SERVICE_AUTHENTICATION}',
                        # 'D':f'{NO_SOMEIP_SD} {WITH_SERVICE_AUTHENTICATION} {WITH_DNSSEC} {WITH_DANE}',
                        # 'E':f'{WITH_SERVICE_AUTHENTICATION} {WITH_CLIENT_AUTHENTICATION}',
                        'F':f'{WITH_SERVICE_AUTHENTICATION} {WITH_CLIENT_AUTHENTICATION} {WITH_ENCRYPTION}',
                        # 'G':f'{WITH_SERVICE_AUTHENTICATION} {WITH_CLIENT_AUTHENTICATION} {WITH_DNSSEC} {WITH_DANE}',
                        'H':f'{WITH_SERVICE_AUTHENTICATION} {WITH_CLIENT_AUTHENTICATION} {WITH_DNSSEC} {WITH_DANE} {WITH_ENCRYPTION}'}

## Read Data

In [24]:
series = dict()
for option, description in compile_definitions.items():
	resultpath = f'{STATISTICS_PARENT_PATH}/{option}-series/'
	# get all files in the resultpath
	resultfilepaths = list(Path(resultpath).rglob('*.csv'))
	# map for runs
	series[option] = dict()
	for filepath in resultfilepaths:
		# Get filename
		filename = filepath.name
		# Get run number
		run_number = int(filename.split('-')[2].split('.')[0].replace('#', ''))
		# Create new dict if not exists
		if run_number not in series[option]:
			series[option][run_number] = dict()
		# Get service id
		service_id = int(filename.split('-')[1])
		# Read csv
		series[option][run_number][service_id] = pd.read_csv(filepath, header=0, sep=',', dtype=np.longlong)


## Select Data

In [ ]:
OPTION = 'ALL'
RUN = 'ALL'
# SERVICE_ID = 300
SERVICE_ID = 'ALL'
total_startup = dict()

def calc_latencies(series, option, run, service_id):
	df = series[option][run][service_id]
	description = compile_definitions[option]
	if NO_SOMEIP_SD in description:
		subscription_start = df['SVCB_SERVICE_RESPONSE_RECEIVE']
	else:
		subscription_start = (df[['OFFER_RECEIVE', 'SVCB_SERVICE_RESPONSE_RECEIVE']].max(axis=1))
	if WITH_SERVICE_AUTHENTICATION in description:
		subscription_end= df['VERIFY_SERVICE_SIGNATURE_END']
	else:
		subscription_end = df['SUBSCRIBE_ACK_RECEIVE']
	# first_offer_to_first_validation
	df['FIRST_OFFER_TO_FIRST_VALIDATION'] = df['VALIDATE_OFFER_START'] - df['OFFER_RECEIVE']
	# setup_delay
	df['SETUP_DELAY'] = subscription_end - df['OFFER_RECEIVE']
	# offer_reaction_time
	df['OFFER_REACTION_TIME'] = df['SUBSCRIBE_SEND'] - df['OFFER_RECEIVE']
	# subscribe_reaction_time
	df['SUBSCRIBE_REACTION_TIME'] = df['SUBSCRIBE_ACK_SEND'] - df['SUBSCRIBE_RECEIVE']
	# subscribe_ack_reaction_time
	df['SUBSCRIBE_ACK_REACTION_TIME'] = subscription_end - df['SUBSCRIBE_ACK_RECEIVE']
	# dns_resolution_time
	df['DNS_RESOLUTION_TIME_SVCB'] = df['SVCB_SERVICE_RESPONSE_RECEIVE'] - df['SVCB_SERVICE_REQUEST_SEND']
	df['DNS_RESOLUTION_TIME_TLSA_CLIENT'] = df['TLSA_CLIENT_RESPONSE_RECEIVE'] - df['TLSA_CLIENT_REQUEST_SEND']
	df['DNS_RESOLUTION_TIME_TLSA_SERVICE'] = df['TLSA_SERVICE_RESPONSE_RECEIVE'] - df['TLSA_SERVICE_REQUEST_SEND']
	# crypto_operations_time
	df['CRYPTO_OPERATIONS_TIME_GENERATE_OFFER_NONCE'] = df['GENERATE_OFFER_NONCE_END'] - df['GENERATE_OFFER_NONCE_START']
	df['CRYPTO_OPERATIONS_TIME_VALIDATE_OFFER'] = df['VALIDATE_OFFER_END'] - df['VALIDATE_OFFER_START']
	df['CRYPTO_OPERATIONS_TIME_CLIENT_SIGN'] = df['CLIENT_SIGN_END'] - df['CLIENT_SIGN_START']
	df['CRYPTO_OPERATIONS_TIME_VERIFY_CLIENT_SIGNATURE'] = df['VERIFY_CLIENT_SIGNATURE_END'] - df['VERIFY_CLIENT_SIGNATURE_START']
	df['CRYPTO_OPERATIONS_TIME_SERVICE_SIGN'] = df['SERVICE_SIGN_END'] - df['SERVICE_SIGN_START']
	df['CRYPTO_OPERATIONS_TIME_VERIFY_SERVICE_SIGNATURE'] = df['VERIFY_SERVICE_SIGNATURE_END'] - df['VERIFY_SERVICE_SIGNATURE_START']
	return subscription_end

if OPTION == 'ALL' and RUN == 'ALL' and SERVICE_ID == 'ALL':
	for option in sorted(series):
		if option not in total_startup:
			total_startup[option] = dict()
		for run in sorted(series[option]):
			if run not in total_startup[option]:
				total_startup[option][run] = dict()
				total_startup[option][run]['OFFER_RECEIVE_MIN'] = np.inf
				total_startup[option][run]['SUBSCRIPTION_END_MAX'] = 0
			for service_id in sorted(series[option][run]):
				subscription_end = calc_latencies(series, option, run, service_id)
				total_startup[option][run]['OFFER_RECEIVE_MIN'] = min(total_startup[option][run]['OFFER_RECEIVE_MIN'], series[option][run][service_id]['OFFER_RECEIVE'].min())
				total_startup[option][run]['SUBSCRIPTION_END_MAX'] = max(total_startup[option][run]['SUBSCRIPTION_END_MAX'], subscription_end.max())
				# progressbar animation
				print(f'\r{option} {run} {service_id}      ', end='')
			total_startup[option][run]['TOTAL_STARTUP_TIME'] = total_startup[option][run]['SUBSCRIPTION_END_MAX'] - total_startup[option][run]['OFFER_RECEIVE_MIN']
	for option in sorted(series):
		# Open txt file for writing
		with open(f'{STATISTICS_PARENT_PATH}/{option}-statistics.txt', 'w') as f:
			all_first_offer_to_first_validation = []
			all_setup_delay = []
			all_offer_reaction_time = []
			all_subscribe_reaction_time = []
			all_subscribe_ack_reaction_time = []
			all_dns_resolution_time_svcb = []
			all_dns_resolution_time_tlsa_client = []
			all_dns_resolution_time_tlsa_service = []
			all_crypto_operations_time_generate_offer_nonce = []
			all_crypto_operations_time_validate_offer = []
			all_crypto_operations_time_client_sign = []
			all_crypto_operations_time_verify_client_signature = []
			all_crypto_operations_time_service_sign = []
			all_crypto_operations_time_verify_service_signature = []
			all_crypto_operations_time_create_signature = []
			all_crypto_operations_time_verify_signature = []
			all_total_startup_time = []
			for run in sorted(series[option]):		
				for service_id in sorted(series[option][run]):
					all_first_offer_to_first_validation.extend(series[option][run][service_id]['FIRST_OFFER_TO_FIRST_VALIDATION'].values / 1000_000)
					all_setup_delay.extend(series[option][run][service_id]['SETUP_DELAY'].values / 1000_000)
					all_offer_reaction_time.extend(series[option][run][service_id]['OFFER_REACTION_TIME'].values / 1000_000)
					all_subscribe_reaction_time.extend(series[option][run][service_id]['SUBSCRIBE_REACTION_TIME'].values / 1000_000)
					all_subscribe_ack_reaction_time.extend(series[option][run][service_id]['SUBSCRIBE_ACK_REACTION_TIME'].values / 1000_000)
					all_dns_resolution_time_svcb.extend(series[option][run][service_id]['DNS_RESOLUTION_TIME_SVCB'].values / 1000_000)
					all_dns_resolution_time_tlsa_client.extend(series[option][run][service_id]['DNS_RESOLUTION_TIME_TLSA_CLIENT'].values / 1000_000)
					all_dns_resolution_time_tlsa_service.extend(series[option][run][service_id]['DNS_RESOLUTION_TIME_TLSA_SERVICE'].values / 1000_000)
					all_crypto_operations_time_generate_offer_nonce.extend(series[option][run][service_id]['CRYPTO_OPERATIONS_TIME_GENERATE_OFFER_NONCE'].values / 1000_000)
					all_crypto_operations_time_validate_offer.extend(series[option][run][service_id]['CRYPTO_OPERATIONS_TIME_VALIDATE_OFFER'].values / 1000_000)
					all_crypto_operations_time_client_sign.extend(series[option][run][service_id]['CRYPTO_OPERATIONS_TIME_CLIENT_SIGN'].values / 1000_000)
					all_crypto_operations_time_verify_client_signature.extend(series[option][run][service_id]['CRYPTO_OPERATIONS_TIME_VERIFY_CLIENT_SIGNATURE'].values / 1000_000)
					all_crypto_operations_time_service_sign.extend(series[option][run][service_id]['CRYPTO_OPERATIONS_TIME_SERVICE_SIGN'].values / 1000_000)
					all_crypto_operations_time_verify_service_signature.extend(series[option][run][service_id]['CRYPTO_OPERATIONS_TIME_VERIFY_SERVICE_SIGNATURE'].values / 1000_000)
				all_total_startup_time.extend([total_startup[option][run]['TOTAL_STARTUP_TIME'] / 1000_000])
			# joint lists for all_crypto_operations_time_create_signature from all_crypto_operations_time_service_sign and all_crypto_operations_time_client_sign
			all_crypto_operations_time_create_signature = all_crypto_operations_time_service_sign + all_crypto_operations_time_client_sign
			# joint lists for all_crypto_operations_time_verify_signature from all_crypto_operations_time_verify_service_signature and all_crypto_operations_time_verify_client_signature
			all_crypto_operations_time_verify_signature = all_crypto_operations_time_verify_service_signature + all_crypto_operations_time_verify_client_signature
			f.write(f"First Offer to First Validation - Min: {np.min(all_first_offer_to_first_validation)}, Mean: {np.mean(all_first_offer_to_first_validation)}, Max: {np.max(all_first_offer_to_first_validation)}\n")
			f.write(f"Setup Delay - Min: {np.min(all_setup_delay)}, Mean: {np.mean(all_setup_delay)}, Max: {np.max(all_setup_delay)}\n")
			f.write(f"Offer Reaction Time - Min: {np.min(all_offer_reaction_time)}, Mean: {np.mean(all_offer_reaction_time)}, Max: {np.max(all_offer_reaction_time)}\n")
			f.write(f"Subscribe Reaction Time - Min: {np.min(all_subscribe_reaction_time)}, Mean: {np.mean(all_subscribe_reaction_time)}, Max: {np.max(all_subscribe_reaction_time)}\n")
			f.write(f"Subscribe Ack Reaction Time - Min: {np.min(all_subscribe_ack_reaction_time)}, Mean: {np.mean(all_subscribe_ack_reaction_time)}, Max: {np.max(all_subscribe_ack_reaction_time)}\n")
			f.write(f"DNS Resolution Time SVCB - Min: {np.min(all_dns_resolution_time_svcb)}, Mean: {np.mean(all_dns_resolution_time_svcb)}, Max: {np.max(all_dns_resolution_time_svcb)}\n")
			f.write(f"DNS Resolution Time TLSA Client - Min: {np.min(all_dns_resolution_time_tlsa_client)}, Mean: {np.mean(all_dns_resolution_time_tlsa_client)}, Max: {np.max(all_dns_resolution_time_tlsa_client)}\n")
			f.write(f"DNS Resolution Time TLSA Service - Min: {np.min(all_dns_resolution_time_tlsa_service)}, Mean: {np.mean(all_dns_resolution_time_tlsa_service)}, Max: {np.max(all_dns_resolution_time_tlsa_service)}\n")
			f.write(f"Crypto Operations Time Generate Offer Nonce - Min: {np.min(all_crypto_operations_time_generate_offer_nonce)}, Mean: {np.mean(all_crypto_operations_time_generate_offer_nonce)}, Max: {np.max(all_crypto_operations_time_generate_offer_nonce)}\n")
			f.write(f"Crypto Operations Time Validate Offer - Min: {np.min(all_crypto_operations_time_validate_offer)}, Mean: {np.mean(all_crypto_operations_time_validate_offer)}, Max: {np.max(all_crypto_operations_time_validate_offer)}\n")
			f.write(f"Crypto Operations Time Client Sign - Min: {np.min(all_crypto_operations_time_client_sign)}, Mean: {np.mean(all_crypto_operations_time_client_sign)}, Max: {np.max(all_crypto_operations_time_client_sign)}\n")
			f.write(f"Crypto Operations Time Verify Client Signature - Min: {np.min(all_crypto_operations_time_verify_client_signature)}, Mean: {np.mean(all_crypto_operations_time_verify_client_signature)}, Max: {np.max(all_crypto_operations_time_verify_client_signature)}\n")
			f.write(f"Crypto Operations Time Service Sign - Min: {np.min(all_crypto_operations_time_service_sign)}, Mean: {np.mean(all_crypto_operations_time_service_sign)}, Max: {np.max(all_crypto_operations_time_service_sign)}\n")
			f.write(f"Crypto Operations Time Verify Service Signature - Min: {np.min(all_crypto_operations_time_verify_service_signature)}, Mean: {np.mean(all_crypto_operations_time_verify_service_signature)}, Max: {np.max(all_crypto_operations_time_verify_service_signature)}\n")
			f.write(f"Crypto Operations Time Create All Signature - Min: {np.min(all_crypto_operations_time_create_signature)}, Mean: {np.mean(all_crypto_operations_time_create_signature)}, Max: {np.max(all_crypto_operations_time_create_signature)}\n")
			f.write(f"Crypto Operations Time Verify All Signature - Min: {np.min(all_crypto_operations_time_verify_signature)}, Mean: {np.mean(all_crypto_operations_time_verify_signature)}, Max: {np.max(all_crypto_operations_time_verify_signature)}\n")
			f.write(f"Total Startup Time - Min: {np.min(all_total_startup_time)}, Mean: {np.mean(all_total_startup_time)}, Max: {np.max(all_total_startup_time)}\n")
elif RUN == 'ALL':
	first_offer_to_first_validation_min_list = list()
	first_offer_to_first_validation_mean_list = list()
	first_offer_to_first_validation_max_list = list()
	a_placeholder_list = list()
	for run in sorted(series[OPTION]):
		calc_latencies(series, OPTION, run, SERVICE_ID)
		first_offer_to_first_validation_min_list.append(series[OPTION][run][SERVICE_ID]['FIRST_OFFER_TO_FIRST_VALIDATION'].min()/1000_000)
		first_offer_to_first_validation_mean_list.append(series[OPTION][run][SERVICE_ID]['FIRST_OFFER_TO_FIRST_VALIDATION'].mean()/1000_000)
		first_offer_to_first_validation_max_list.append(series[OPTION][run][SERVICE_ID]['FIRST_OFFER_TO_FIRST_VALIDATION'].max()/1000_000)
		a_placeholder_list.append(series[OPTION][run][SERVICE_ID]['SETUP_DELAY'].min()/1000_000)
	print(a_placeholder_list)
	print(np.min(a_placeholder_list))
	print(np.mean(a_placeholder_list))
	print(np.max(a_placeholder_list))
else:
	calc_latencies(series, OPTION, RUN, SERVICE_ID)


H 24 7013